In [1]:
import boto3
import json
import time
import os
from pathlib import Path

# Configuration
APPLICATION_ID = "graphrag-pdf"
REGION = "us-east-1"  # Change this to your preferred region
PROVISIONED_MEMORY = "16"

# Initialize AWS clients
neptune = boto3.client("neptune-graph", region_name=REGION)
opensearch = boto3.client("opensearchserverless", region_name=REGION)
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Setting up GraphRAG toolkit in region: {REGION}")
print(f"Application ID: {APPLICATION_ID}")


Setting up GraphRAG toolkit in region: us-east-1
Application ID: graphrag-pdf


In [2]:
print("Creating IAM role and policies...")

try:
    # Initialize IAM client
    iam = boto3.client('iam', region_name=REGION)
    
    # Create IAM role
    role_name = f"{APPLICATION_ID}-role"
    
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": [
                        "neptune-graph.amazonaws.com",
                        "aoss.amazonaws.com",
                        "bedrock.amazonaws.com"
                    ]
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }
    
    try:
        role_response = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(assume_role_policy),
            Description=f"Role for GraphRAG toolkit {APPLICATION_ID}"
        )
        print(f"✅ IAM role created: {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        print(f"ℹ️ IAM role already exists: {role_name}")
        role_response = iam.get_role(RoleName=role_name)
    
    role_arn = role_response['Role']['Arn']
    
    # Create policy for Neptune Analytics
    neptune_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "neptune-graph:*"
                ],
                "Resource": f"arn:aws:neptune-graph:{REGION}:*:graph/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for OpenSearch Serverless
    opensearch_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "aoss:*"
                ],
                "Resource": f"arn:aws:aoss:{REGION}:*:collection/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for Bedrock
    bedrock_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-haiku-20240307-v1",
                    f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2"
                ]
            }
        ]
    }
    
    # Attach policies to role
    for policy_name, policy_doc in [
        ("neptune", neptune_policy),
        ("opensearch", opensearch_policy),
        ("bedrock", bedrock_policy)
    ]:
        policy_name = f"{APPLICATION_ID}-{policy_name}-policy"
        try:
            iam.put_role_policy(
                RoleName=role_name,
                PolicyName=policy_name,
                PolicyDocument=json.dumps(policy_doc)
            )
            print(f"✅ Attached policy: {policy_name}")
        except Exception as e:
            print(f"⚠️ Error attaching policy {policy_name}: {e}")
            raise
    
    print(f"\n✅ IAM setup complete!")
    print(f"Role ARN: {role_arn}")
    
except Exception as e:
    print(f"❌ Error in IAM setup: {e}")
    raise


Creating IAM role and policies...
✅ IAM role created: graphrag-pdf-role
✅ Attached policy: graphrag-pdf-neptune-policy
✅ Attached policy: graphrag-pdf-opensearch-policy
✅ Attached policy: graphrag-pdf-bedrock-policy

✅ IAM setup complete!
Role ARN: arn:aws:iam::533267284022:role/graphrag-pdf-role


In [3]:
print("Creating Neptune Analytics Graph...")

# Create Neptune Graph
graph_response = neptune.create_graph(
    graphName=f"{APPLICATION_ID}-graph",
    provisionedMemory=int(PROVISIONED_MEMORY),
    deletionProtection=False,
    publicConnectivity=True,
    replicaCount=1,
    vectorSearchConfiguration={
        "dimension": 1024
    }
)

# Extract ID and endpoint from the response
graph_id = graph_response["id"]
graph_endpoint = graph_response["endpoint"]

print(f"✅ Neptune Graph created successfully!")
print(f"   Graph ID: {graph_id}")
print(f"   Endpoint: {graph_endpoint}")
print(f"   Status: {graph_response['status']}")

# Wait for graph to be available
print("Waiting for graph to be available...")
waiter = neptune.get_waiter("graph_available")
waiter.wait(graphIdentifier=graph_id)
print("✅ Graph is now available!")


Creating Neptune Analytics Graph...
✅ Neptune Graph created successfully!
   Graph ID: g-pyproxlgo4
   Endpoint: g-pyproxlgo4.us-east-1.neptune-graph.amazonaws.com
   Status: CREATING
Waiting for graph to be available...
✅ Graph is now available!


In [4]:
print("Creating OpenSearch security policies...")

try:
    # Create encryption policy FIRST (required before collection creation)
    encryption_policy = {
        "Rules": [
            {
                "Resource": [f"collection/{APPLICATION_ID}-collection"],
                "ResourceType": "collection"
            }
        ],
        "AWSOwnedKey": True
    }
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-encryption",
            policy=json.dumps(encryption_policy),
            type="encryption"
        )
        print("✅ Encryption policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Encryption policy already exists")
    
    # Create network policy
    network_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "dashboard"
                },
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "collection"
                }
            ],
            "AllowFromPublic": True
        }
    ]
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-network",
            policy=json.dumps(network_policy),
            type="network"
        )
        print("✅ Network policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Network policy already exists")
    
    # Create access policy
    # Get current user ARN
    sts = boto3.client("sts")
    user_arn = sts.get_caller_identity()["Arn"]
    
    access_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "Permission": [
                        "aoss:CreateCollectionItems",
                        "aoss:DeleteCollectionItems",
                        "aoss:UpdateCollectionItems",
                        "aoss:DescribeCollectionItems"
                    ],
                    "ResourceType": "collection"
                },
                {
                    "Resource": [f"index/{APPLICATION_ID}-collection/*"],
                    "Permission": [
                        "aoss:CreateIndex",
                        "aoss:DeleteIndex",
                        "aoss:UpdateIndex",
                        "aoss:DescribeIndex",
                        "aoss:ReadDocument",
                        "aoss:WriteDocument"
                    ],
                    "ResourceType": "index"
                }
            ],
            "Principal": [user_arn]
        }
    ]
    
    try:
        opensearch.create_access_policy(
            name=f"{APPLICATION_ID}-access",
            policy=json.dumps(access_policy),
            type="data"
        )
        print("✅ Access policy created")
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Access policy already exists")
    
    print("✅ All security policies are ready!")
    
except Exception as e:
    print(f"❌ Error creating security policies: {e}")
    raise

# Create OpenSearch Collection
print("\nCreating OpenSearch Serverless Collection...")

try:
    # Create collection
    try:
        collection_response = opensearch.create_collection(
            name=f"{APPLICATION_ID}-collection",
            type="VECTORSEARCH",
            standbyReplicas="DISABLED"
        )
        
        collection_id = collection_response["createCollectionDetail"]["id"]
        print(f"✅ OpenSearch Collection created successfully!")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_response['createCollectionDetail']['status']}")
        
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Collection already exists, retrieving details...")
        collection_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = collection_response["collectionDetails"][0]
        collection_id = collection_detail["id"]
        
        print(f"✅ Using existing OpenSearch Collection:")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_detail['status']}")
    
    # Wait for collection to be active and get the endpoint
    print("Waiting for collection to be active...")
    while True:
        status_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = status_response["collectionDetails"][0]
        status = collection_detail["status"]
        print(f"   Collection status: {status}")
        
        if status == "ACTIVE":
            collection_endpoint = collection_detail["collectionEndpoint"]
            print(f"   Collection endpoint: {collection_endpoint}")
            break
        elif status in ["FAILED", "DELETED"]:
            raise Exception(f"Collection creation failed with status: {status}")
        
        time.sleep(10)
    
    print("✅ Collection is now active!")
    
except Exception as e:
    print(f"❌ Error creating OpenSearch Collection: {e}")
    raise


Creating OpenSearch security policies...
✅ Encryption policy created
✅ Network policy created
✅ Access policy created
✅ All security policies are ready!

Creating OpenSearch Serverless Collection...
✅ OpenSearch Collection created successfully!
   Collection ID: lr5sfuwpmthmxio8auo7
   Status: CREATING
Waiting for collection to be active...
   Collection status: CREATING
   Collection status: CREATING
   Collection status: CREATING
   Collection status: ACTIVE
   Collection endpoint: https://lr5sfuwpmthmxio8auo7.us-east-1.aoss.amazonaws.com
✅ Collection is now active!


In [5]:
# Create .env file with configuration
env_content = f"""# GraphRAG Toolkit Configuration
APPLICATION_ID={APPLICATION_ID}
AWS_REGION={REGION}

# Neptune Analytics
NEPTUNE_GRAPH_ID={graph_id}
NEPTUNE_GRAPH_ENDPOINT={graph_endpoint}

# OpenSearch Serverless
OPENSEARCH_COLLECTION_ID={collection_id}
OPENSEARCH_COLLECTION_ENDPOINT={collection_endpoint}

# Bedrock Models
BEDROCK_EMBEDDING_MODEL=amazon.titan-embed-text-v2
BEDROCK_LLM_MODEL=anthropic.claude-3-haiku-20240307-v1
"""

# Save to .env file
env_path = Path(".env")
env_path.write_text(env_content)

print("✅ Configuration saved to .env file")
print("\nSetup complete! You can now proceed with the other notebooks.")


✅ Configuration saved to .env file

Setup complete! You can now proceed with the other notebooks.
